In [5]:
import requests
import os
from typing import List
import pickle
import zipfile
import rasterio
from affine import Affine
import pyproj
import numpy as np
import scipy.ndimage
from rasterio.warp import reproject, Resampling
import PIL
import matplotlib.pyplot as plt
from base64 import b64encode
import tables
from pathlib import Path
try:
    from StringIO import StringIO
    py3 = False
except ImportError:
    from io import StringIO, BytesIO
    py3 = True
from ipyleaflet import Map, ImageOverlay, basemap_to_tiles, basemaps, LayerGroup, FullScreenControl, Marker, Popup, LayersControl, LegendControl
from ipywidgets import HTML

from matplotlib.colors import ListedColormap
import matplotlib.colors as mcolors

# Step 1: List of hex color values
hex_colors = ['#000000', '#006400', '#ffbb22', '#ffff4c', '#f096ff', '#fa0000', '#b4b4b4', '#f0f0f0','#0064c8', '#0096a0', '#00cf75', '#fae6a0']  # ESA world cover

# Step 2: Convert hex to RGB
rgb_colors = [mcolors.hex2color(color) for color in hex_colors]

# Step 3: Create a LinearSegmentedColormap
wc_colormap = mcolors.ListedColormap(rgb_colors, name="esa_world_cover")

## Decoding RasterSpec

In [8]:
def decode_spec(spec_arr):
    spec_bytes = spec_arr.tobytes()
    return pickle.loads(spec_bytes)

## The original data is in the WGS 84 projection, but Leaflet uses Web Mercator, so we need to reproject.

In [9]:
def reproject_to_mercator(arrs, spec):
    with rasterio.Env():
        if not isinstance(arrs, List):
            arrs = [arrs]
        rows, cols = arrs[0].shape[-2:]
        src_transform = spec.transform
        src_crs = {'init': f'EPSG:{spec.epsg}'}

        dst_crs = {'init': 'EPSG:3857'}
        dst_transform, width, height = rasterio.warp.calculate_default_transform(src_crs, dst_crs, cols, rows, *spec.bounds)
        reprojected = []
        for arr in arrs:
            if len(arr.shape) > 2:
                dst_shape = arr.shape[0], height, width
            else: 
                dst_shape = height, width
            destination = np.zeros(dst_shape)

            reproject(
                arr,
                destination,
                src_transform=src_transform,
                src_crs=src_crs,
                dst_transform=dst_transform,
                dst_crs=dst_crs,
                resampling=Resampling.nearest)
            reprojected.append(destination)
    return reprojected

## convert NumPy array to an image.

In [10]:
def to_PIL(arr, minv=0, maxv=None, cm_func=None):
    if arr.shape[0] == 3: # RGB
        arr = arr.transpose((1,2,0))
        arr_mask = np.where(np.isfinite(arr[:,:,0]), 255, 0)
    else: # wc or slope
        arr_mask = np.where(np.isfinite(arr), 255, 0)
    arr_norm = (arr - minv) / (maxv - minv)
    arr_norm = np.where(np.isfinite(arr), arr_norm, 0)
    if len(arr.shape) == 2:
        arr_norm = cm_func(arr_norm)
    arr_im = PIL.Image.fromarray(np.uint8(arr_norm*255))
    mask = PIL.Image.fromarray(np.uint8(arr_mask), mode='L')
    im = PIL.Image.new('RGBA', arr_norm.shape[:2], color=None)
    im.paste(arr_im, mask=mask)
    return im

## Generate Image url

In [11]:
def get_url(img, py3=None):
    if py3:
        f = BytesIO()
    else:
        f = StringIO()
    img.save(f, 'png')
    data = b64encode(f.getvalue())
    if py3:
        data = data.decode('ascii')
    imgurl = 'data:image/png;base64,' + data
    return imgurl

In [12]:
def reproject_bounds(raster_spec, crs_to='EPSG:4326'):
    transformer = pyproj.Transformer.from_crs(f'EPSG:{raster_spec.epsg}', crs_to, always_xy=True)
    # Transform the bounds
    minx, miny = transformer.transform(raster_spec.bounds[0], raster_spec.bounds[1])
    maxx, maxy = transformer.transform(raster_spec.bounds[2], raster_spec.bounds[3])
    return ((miny,minx), (maxy, maxx))

In [13]:
def create_base_map(center = [-10, -60], zoom=2):
    m = Map(center=center, zoom=zoom, interpolation='nearest', scroll_wheel_zoom=True)
    m.layout.height = '800px'
    tile = basemap_to_tiles(basemaps.Esri.WorldImagery)
    m.add(tile)
    m.add(FullScreenControl)
    return m

In [14]:
def create_img_overlay(data, partition_number=0, year=2019):
    partition = data.root[f'{year}/{partition_number}/input'][:, [1,2,3,13]]
    s2_arr = partition[:, :3]
    wc_arr = partition[:, -1]
    slope_arr = data.root[f'{year}/{partition_number}/slope']
    spec_arr = data.root[f'{year}/{partition_number}/spec']
    loc_arr = data.root[f'{year}/{partition_number}/gedi_attrs'][:, 10:12]
    dc_arr = data.root[f'{year}/{partition_number}/defective_cover']
    dd_arr = data.root[f'{year}/{partition_number}/delta_day']
    s2_ios, wc_ios, slope_ios, loc_makers = [],[],[],[]
    for i, (s2, wc, slope) in enumerate(zip(s2_arr, wc_arr, slope_arr)):
        spec = decode_spec(spec_arr[i])
        s2, wc, slope = reproject_to_mercator([s2, wc, slope], spec)
        bounds = reproject_bounds(spec)
        img_overlays = []
        s2_img = to_PIL(s2, np.nanmin(s2), np.nanmax(s2))
        wc_img = to_PIL(wc, 0, 100, wc_colormap)
        slop_img = to_PIL(slope, 0, 90, plt.cm.gist_earth)
        for im in [s2_img, wc_img, slop_img]:
            imgurl = get_url(im, py3=py3)
            io = ImageOverlay(url=imgurl, bounds=bounds)
            img_overlays.append(io)
        s2_ios.append(img_overlays[0])
        wc_ios.append(img_overlays[1])
        slope_ios.append(img_overlays[2])
        marker = Marker(location=loc_arr[i].tolist())
        message = HTML()
        message.value = f"defective cover: {dc_arr[i]} <br>delta days: {dd_arr[i]} "
        marker.popup = message
        loc_makers.append(marker)
    return s2_ios, wc_ios, slope_ios, loc_makers

In [15]:
data_dir = 'data/GEDI/vis'
zone = '01G'
h5_file = Path.home() / data_dir/ f'{zone}.h5'
data = tables.open_file(h5_file)

In [ ]:
layers_list = create_img_overlay(data, 7)

In [17]:
m = create_base_map()
for i, layers in enumerate(layers_list):
    group = LayerGroup(name=f'group{i}',layers=layers)
    m.add(group)
control = LayersControl(position='topright')
m.add(control)
# overlay_group.interact(opacity=(0.0,1.0,0.01))

Map(center=[-10, -60], controls=(ZoomControl(options=['position', 'zoom_in_text', 'zoom_in_title', 'zoom_out_t…

In [18]:
m.layout.height = '800px'

In [19]:
# import xarray as xr
# ds = xr.open_dataset(h5_file, group='2019/0', engine='h5netcdf')

In [20]:
# ds.delta_day